<a href="https://colab.research.google.com/github/simranshika29/data_science_capstone/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn gensim nltk transformers torch tqdm matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 38.5 MB/s eta 0:00:00


In [8]:
import os
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from gensim.models import Word2Vec

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from transformers import BertTokenizer, BertModel
import torch

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Explicitly download punkt_tab

STOPWORDS = set(stopwords.words('english'))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


device(type='cpu')

In [5]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("/content/IMDB Dataset (1).csv")
df = df[['review', 'sentiment']].dropna()

df['label'] = df['sentiment'].map({'negative':0, 'positive':1})
df.head()

Saving IMDB Dataset.csv to IMDB Dataset (1).csv


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [9]:
def clean_text(text):
    text = re.sub(r"<.*?>"," ", text)
    text = re.sub(r"[^a-zA-Z']"," ", text)
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)

df['clean_review'] = df['review'].apply(clean_text)

X = df['clean_review'].values
y = df['label'].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
import os
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from gensim.models import Word2Vec

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from transformers import BertTokenizer, BertModel
import torch

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to download the missing resource

STOPWORDS = set(stopwords.words('english'))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


device(type='cpu')

In [11]:
bow_vectorizer = CountVectorizer(max_features=20000)
X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow = bow_vectorizer.transform(X_test_text)


In [12]:
tfidf_vectorizer = TfidfVectorizer(max_features=20000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

In [13]:
tfidf_ngram = TfidfVectorizer(ngram_range=(1,2), max_features=40000)
X_train_ngram = tfidf_ngram.fit_transform(X_train_text)
X_test_ngram = tfidf_ngram.transform(X_test_text)

In [14]:
def train_model(X_train, X_test, y_train, y_test, name):
      clf = LogisticRegression(max_iter=2000)
      clf.fit(X_train, y_train)
      pred = clf.predict(X_test)
      acc = accuracy_score(y_test, pred)
      print(f"\n===== {name} =====")
      print("Accuracy:", acc)
      print(classification_report(y_test, pred))
      return clf, acc

In [15]:
results = {}

bow_clf, results['BoW'] = train_model(X_train_bow, X_test_bow, y_train, y_test, "BOW")
tfidf_clf, results['TF-IDF'] = train_model(X_train_tfidf, X_test_tfidf, y_train, y_test, "TF-IDF")
ngram_clf, results['TF-IDF NGRAM'] = train_model(X_train_ngram, X_test_ngram, y_train, y_test, "TF-IDF NGRAM")

results


===== BOW =====
Accuracy: 0.8857
              precision    recall  f1-score   support

           0       0.89      0.88      0.89      5000
           1       0.88      0.89      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


===== TF-IDF =====
Accuracy: 0.8972
              precision    recall  f1-score   support

           0       0.90      0.89      0.90      5000
           1       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000


===== TF-IDF NGRAM =====
Accuracy: 0.9028
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      5000
           1       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg  

{'BoW': 0.8857, 'TF-IDF': 0.8972, 'TF-IDF NGRAM': 0.9028}

In [16]:
def show_top_words(vectorizer, clf, top_n=20):
      feature_names = np.array(vectorizer.get_feature_names_out())
      coefs = clf.coef_[0]

      top_pos = feature_names[np.argsort(coefs)][-top_n:]
      top_neg = feature_names[np.argsort(coefs)][:top_n]

      print("\nTop Positive Words:\n", top_pos)
      print("\nTop Negative Words:\n", top_neg)

show_top_words(tfidf_vectorizer, tfidf_clf)


Top Positive Words:
 ['fantastic' 'definitely' 'superb' 'enjoyable' 'love' 'today' 'highly'
 'well' 'fun' 'brilliant' 'favorite' 'enjoyed' 'hilarious' 'loved'
 'wonderful' 'amazing' 'best' 'perfect' 'excellent' 'great']

Top Negative Words:
 ['worst' 'bad' 'waste' 'awful' 'boring' 'poor' 'terrible' 'nothing' 'dull'
 'poorly' 'horrible' 'worse' 'fails' 'disappointing' 'disappointment'
 'minutes' 'instead' 'unfortunately' 'stupid' 'supposed']


In [17]:
sentences = [s.split() for s in X_train_text]

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=5
)

In [ ]:
from google.colab import files
files.upload()


In [ ]:
glove = {}

with open("glove.6B.100d.txt", 'r', encoding='utf8') as f:
    for line in tqdm(f):
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove[word] = vector

In [ ]:
def glove_vector(sentence):
      words = sentence.split()
      vecs = [glove[w] for w in words if w in glove]
      if not vecs:
              return np.zeros(100)
      return np.mean(vecs, axis=0)

X_train_glove = np.vstack([glove_vector(s) for s in X_train_text])
X_test_glove = np.vstack([glove_vector(s) for s in X_test_text])

In [ ]:
glove_clf, results['GloVe'] = train_model(X_train_glove, X_test_glove, y_train, y_test, "GLOVE")
results


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased").to(DEVICE)
bert.eval()


In [ ]:
def bert_embed(text_list, batch_size=16):
      all_vecs = []

      for i in tqdm(range(0, len(text_list), batch_size)):
              batch = text_list[i:i+batch_size]
              enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
              ids = enc['input_ids'].to(DEVICE)
              att = enc['attention_mask'].to(DEVICE)

              with torch.no_grad():
                          out = bert(ids, attention_mask=att)
                          cls_vec = out.last_hidden_state[:,0,:].cpu().numpy()

              all_vecs.append(cls_vec)

      return np.vstack(all_vecs)

X_train_bert = bert_embed(X_train_text)
X_test_bert = bert_embed(X_test_text)

In [ ]:
bert_clf, results['BERT'] = train_model(X_train_bert, X_test_bert, y_train, y_test, "BERT CLS")
results



In [ ]:
df_results = pd.DataFrame(list(results.items()), columns=["Model","Accuracy"])
df_results.sort_values("Accuracy", ascending=False)

